# Evaluator Module
The Evaluator module creates evaluation reports.

Reports contain evaluation metrics depending on models specified in the evaluation config.

In [2]:
# reloads modules automatically before entering the execution of code
%load_ext autoreload
%autoreload 2

# third parties imports
import numpy as np 
import pandas as pd
# -- add new imports here --
from surprise import accuracy
from surprise import model_selection
from surprise import Reader
from surprise import Dataset
from surprise import SVD
 
# local imports
from configs import EvalConfig
from constants import Constant as C
from loaders import export_evaluation_report
from loaders import load_ratings
from models import get_top_n
# -- add new imports here --

# 1. Model validation functions
Validation functions are a way to perform crossvalidation on recommender system models. 

In [3]:
def generate_split_predictions(algo, ratings_dataset, eval_config):
    trainset, testset = model_selection.train_test_split(ratings_dataset, test_size=eval_config.test_size)
    algo.fit(trainset)
    predictions = algo.test(testset)
    return predictions


def generate_loo_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user on a random Leave-one-out split (LOO)"""
    # -- implement the function generate_loo_top_n --
    loo = model_selection.LeaveOneOut(n_splits=1, random_state=1)
    trainset, testset = next(loo.split(ratings_dataset))
    algo.fit(trainset)
    anti_test_set = trainset.build_anti_testset()
    predictions = algo.test(anti_test_set)
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    

    return anti_testset_top_n, testset


def generate_full_top_n(algo, ratings_dataset, eval_config):
    """Generate top-n recommendations for each user with full training set (LOO)"""
    # -- implement the function generate_full_top_n --
    full_train_set = ratings_dataset.build_full_trainset()
    algo.fit(full_train_set)
    anti_test_set = full_train_set.build_anti_testset()
    predictions = algo.test(anti_test_set)
    anti_testset_top_n = get_top_n(predictions, n=eval_config.top_n_value)
    return anti_testset_top_n


def precompute_information():
    """ Returns a dictionary that precomputes relevant information for evaluating in full mode
    
    Dictionary keys:
    - precomputed_dict["item_to_rank"] : contains a dictionary mapping movie ids to rankings
    - (-- for your project, add other relevant information here -- )
    """
    precomputed_dict = {}
    precomputed_dict["item_to_rank"] = None
    return precomputed_dict                


def create_evaluation_report(eval_config, sp_ratings, precomputed_dict, available_metrics):
    """ Create a DataFrame evaluating various models on metrics specified in an evaluation config.  
    """
    evaluation_dict = {}
    for model_name, model, arguments in eval_config.models:
        print(f'Handling model {model_name}')
        algo = model(**arguments)
        evaluation_dict[model_name] = {}
        
        # Type 1 : split evaluations
        if len(eval_config.split_metrics) > 0:
            print('Training split predictions')
            predictions = generate_split_predictions(algo, sp_ratings, eval_config)
            for metric in eval_config.split_metrics:
                print(f'- computing metric {metric}')
                assert metric in available_metrics['split']
                evaluation_function, parameters =  available_metrics["split"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(predictions, **parameters) 

        # Type 2 : loo evaluations
        if len(eval_config.loo_metrics) > 0:
            print('Training loo predictions')
            anti_testset_top_n, testset = generate_loo_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.loo_metrics:
                assert metric in available_metrics['loo']
                evaluation_function, parameters =  available_metrics["loo"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(anti_testset_top_n, testset, **parameters)
        
        # Type 3 : full evaluations
        if len(eval_config.full_metrics) > 0:
            print('Training full predictions')
            anti_testset_top_n = generate_full_top_n(algo, sp_ratings, eval_config)
            for metric in eval_config.full_metrics:
                assert metric in available_metrics['full']
                evaluation_function, parameters =  available_metrics["full"][metric]
                evaluation_dict[model_name][metric] = evaluation_function(
                    anti_testset_top_n,
                    **precomputed_dict,
                    **parameters
                )
        
    return pd.DataFrame.from_dict(evaluation_dict).T

# 2. Evaluation metrics
Implement evaluation metrics for either rating predictions (split metrics) or for top-n recommendations (loo metric, full metric)

In [7]:
def get_hit_rate(anti_testset_top_n, testset):
    """Compute the average hit over the users (loo metric)
    
    A hit (1) happens when the movie in the testset has been picked by the top-n recommender
    A fail (0) happens when the movie in the testset has not been picked by the top-n recommender
    """
    # -- implement the function get_hit_rate --
    return hit_rate


def get_novelty(anti_testset_top_n, item_to_rank):
    """Compute the average novelty of the top-n recommendation over the users (full metric)
    
    The novelty is defined as the average ranking of the movies recommended
    """
    # -- implement the function get_novelty --
    return average_rank_sum

# 3. Evaluation workflow
Load data, evaluate models and save the experimental outcomes

In [ ]:
AVAILABLE_METRICS = {
    "split": {
        "mae": (accuracy.mae, {'verbose': False}),
        # -- add new split metrics here --
    },
    # -- add new types of metrics here --
}

sp_ratings = load_ratings(surprise_format=True)
algo = SVD()
test = generate_split_predictions(algo, sp_ratings, EvalConfig)
top_n_loo_top,test_set_loo = generate_loo_top_n(algo, sp_ratings, EvalConfig)
rows = []
for user_id, item_list in top_n_loo_top.items():
    for item_id, estimated_rating in item_list:
        rows.append((user_id, item_id, estimated_rating))

# Création du DataFrame
df_topn = pd.DataFrame(rows, columns=['user', 'item', 'estimated_rating'])
print(df_topn.head(50))
top_n_full = generate_full_top_n(algo, sp_ratings, EvalConfig) 
print(top_n_full[0])
precomputed_dict = precompute_information()
evaluation_report = create_evaluation_report(EvalConfig, sp_ratings, precomputed_dict, AVAILABLE_METRICS)
export_evaluation_report(evaluation_report)



defaultdict(<class 'list'>, {1: [(50, 3.747856146214743), (2959, 3.7047887964124837), (908, 3.6922387399309624), (913, 3.682633551374087), (4226, 3.6499262466191054), (969, 3.6377443065956956), (858, 3.635862406843461), (1221, 3.617142813062875), (903, 3.6151732985775173), (3683, 3.6114108243066387), (2571, 3.5868823639736354), (307, 3.5807903588194367), (1247, 3.580253901577206), (1254, 3.576761715268264), (1228, 3.570091624563418), (905, 3.5644540264439515), (7502, 3.5641293770876956), (8132, 3.563008776583434), (1219, 3.5617873247854885), (527, 3.55509941242652), (58559, 3.551101376074509), (1193, 3.536731473840808), (2542, 3.5304630029657367), (1212, 3.5276888027669524), (923, 3.523014553105126), (954, 3.5202323507299207), (293, 3.520001041399641), (1299, 3.5161988043611148), (926, 3.5144277033342464), (6787, 3.5122759976006455), (1252, 3.5100677998517695), (3088, 3.5065574655929748), (3462, 3.504659132679842), (1945, 3.504158301667462), (1204, 3.4977813472932544), (48780, 3.496782